Time Series Plot of H-bond potential for IM30

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------
# Folder for IM30
# --------------------------
folder_path = "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/"

# Color palette for multiple bias curves
colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#46f0f0',
          '#f032e6', '#bcf60c', '#fabebe', '#008080', '#e6beff', '#9a6324']

# --------------------------
# Create figure
# --------------------------
plt.figure(figsize=(14, 8))
plt.title("IM30", fontsize=30, fontweight='bold')
plt.xlabel("Time (μs)", fontsize=30, fontweight='bold')
plt.ylabel("H-Bond Potential (KT)", fontsize=30, fontweight='bold')

# List and sort log files by bias
log_files = [f for f in os.listdir(folder_path) if f.startswith("log_bias_") and f.endswith(".log")]
log_files.sort(key=lambda x: float(x.split('_')[2].replace('.log', '')))

# Compute max bias to normalize
bias_values_raw = [float(f.split('_')[2].replace('.log','')) for f in log_files]
max_bias = max(bias_values_raw)

# Loop through files and plot each bias
for j, file in enumerate(log_files):
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path, delim_whitespace=True)
        df.columns = df.columns.str.strip().str.rstrip('.')  # Clean column names
        
        if "Simulation.timestep" not in df.columns:
            continue
        if "H-Bond_Potential" in df.columns:
            hbond_col = "H-Bond_Potential"
        elif "H-Bond_Potential." in df.columns:
            hbond_col = "H-Bond_Potential."
        else:
            continue

        df['Simulation.timestep'] = pd.to_numeric(df['Simulation.timestep'], errors='coerce')
        df[hbond_col] = pd.to_numeric(df[hbond_col], errors='coerce')
        
        # Filter timesteps between 1e9 and 2e9 (10–20 μs)
        df_filtered = df[(df['Simulation.timestep'] >= 1e9) & (df['Simulation.timestep'] <= 2e9)]
        if df_filtered.empty:
            continue

        # Convert timesteps to μs
        time_us = df_filtered['Simulation.timestep'] / 1e8  # 1e9 → 10 μs

        # Compute bias as % H-bond strength
        bias_val = float(file.split('_')[2].replace('.log',''))
        bias_percent = (bias_val / max_bias) * 100

        # Plot line with unique color
        plt.plot(time_us, df_filtered[hbond_col],
                 color=colors[j % len(colors)],
                 label=f"{bias_percent:.0f}%")
        
    except Exception as e:
        print(f"Error processing {file}: {e}")

# Styling axes and ticks
ax = plt.gca()
ax.set_xlim(10, 20)
ax.tick_params(axis='both', labelsize=26, width=2, length=6)  # bold ticks
ax.spines['top'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

# Small legend, bold text
legend = plt.legend(title="Bias (%)", fontsize=14, title_fontsize=14, frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')
legend.get_title().set_fontweight('bold')

plt.tight_layout()

# Save figure
output_path = "IM30_hbond_timeseries_10_20us_styled_smalllegend.png"
plt.savefig(output_path, dpi=600)
print(f"Time series figure saved as {output_path}")

plt.show()


Average H-Bond Vs HBS Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ---- Folders and labels ----
folders = {
    "IM30": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/",
    "IM30 H0-3": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/3de12d848c06a70b5668c64369550562/",
    "IM30 H4-6": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/7fd5f713b9bd4167bafd3d2d9378bc51/"
}

# Colors for datasets
colors = {
    "IM30": "red",
    "IM30 H0-3": "green",
    "IM30 H4-6": "blue"
}

plt.figure(figsize=(12, 8))
plt.xlabel("H-bond Strength (%)", fontsize=30, fontweight='bold')
plt.ylabel("Average H-Bond Potential (KT)", fontsize=30, fontweight='bold')
plt.xticks(fontsize=26, fontweight='bold')
plt.yticks(fontsize=26, fontweight='bold')

for label, folder_path in folders.items():
    # Locate log files
    log_files = [f for f in os.listdir(folder_path) if f.startswith("log_bias_") and f.endswith(".log")]
    log_files.sort(key=lambda x: float(x.split('_')[2].replace('.log', '')))

    raw_bias_values = []
    avg_hbond_values = []
    std_hbond_values = []

    for file in log_files:
        bias_str = file.split('_')[2].replace('.log', '')
        bias_val = float(bias_str)
        file_path = os.path.join(folder_path, file)

        raw_bias_values.append(bias_val)

        try:
            df = pd.read_csv(file_path, delim_whitespace=True)
            df.columns = df.columns.str.strip().str.rstrip('.')  # Clean column names

            if "Simulation.timestep" not in df.columns:
                continue

            # Identify H-bond column
            hbond_col_candidates = ["H-Bond_Potential", "H-Bond_Potential."]
            hbond_col = next((c for c in hbond_col_candidates if c in df.columns), None)
            if hbond_col is None:
                continue

            df['Simulation.timestep'] = pd.to_numeric(df['Simulation.timestep'], errors='coerce')
            df[hbond_col] = pd.to_numeric(df[hbond_col], errors='coerce')

            # Filter timesteps (1e9 → 2e9)
            df_filtered = df[(df['Simulation.timestep'] >= 1_000_000_000) &
                             (df['Simulation.timestep'] <= 2_000_000_000)]

            if not df_filtered.empty:
                avg_hbond_values.append(df_filtered[hbond_col].mean())
                std_hbond_values.append(df_filtered[hbond_col].std() / np.sqrt(12))  # divide by sqrt(12)

        except Exception as e:
            print(f"Error processing {file}: {e}")

    # Normalize bias → 0–100%
    if raw_bias_values:
        max_bias = max(raw_bias_values)
        bias_percent = [(b / max_bias) * 100 for b in raw_bias_values]

        plt.errorbar(
            bias_percent,
            avg_hbond_values,
            yerr=std_hbond_values,
            marker='o',
            linewidth=2,
            capsize=4,
            label=label,
            color=colors[label]
        )

# Bold legend
legend = plt.legend(title="", fontsize=26, title_fontsize=26, ncol=1)
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()

# Save figure
output_path = "hbond_bias_strength_bold.png"
plt.savefig(output_path, dpi=300)
print(f"Figure saved as {output_path}")

plt.show()


Replica_acceptance_rate_IM30

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---- Load data ----
file_path = "/home/POLY/bhandarit/Desktop/chain_n_chain/replica_h_bond/replica_exchange_rate_IM30.log"

# Adjust separator if needed; here using tab
df = pd.read_csv(file_path, sep='\t')

# ---- Set up plot ----
plt.figure(figsize=(12, 8))
sns.set(style="whitegrid")

# Get sorted replica indices
replica_pairs = sorted(df['Replica_i'].unique())

# Plot KDE for each replica pair
for rep in replica_pairs:
    pair_df = df[df['Replica_i'] == rep]
    sns.kdeplot(
        pair_df['Acceptance_Rate'], 
        label=f"{rep}-{rep+1}", 
        fill=True, 
        linewidth=2
    )

# ---- Styling ----
plt.xlabel("Acceptance Rate", fontsize=30, fontweight='bold')
plt.ylabel("Density", fontsize=30, fontweight='bold')
plt.xticks(fontsize=26, fontweight='bold')
plt.yticks(fontsize=26, fontweight='bold')

plt.legend(
    title="Replica Pair", 
    title_fontsize=26, 
    fontsize=26,
    frameon=False,
    loc="upper right"
)

plt.grid(False)
plt.tight_layout()

# ---- Save figure ----
output_path = "replica_acceptance_rate_IM30.png"
plt.savefig(output_path, dpi=600)
print(f"Figure saved as {output_path}")

plt.show()
